In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score

# 1. Load and clean the data
file_path = '../data/time_binned_dataset.csv'
print(f"Loading data: {file_path}")
df = pd.read_csv(file_path)

# Convert datetime and set as index
df['datetime'] = pd.to_datetime(df['datetime'])
df.set_index('datetime', inplace=True)

# Drop missing values
df = df.dropna()
print(f"After dropping missing values, the dataset has {len(df)} samples remaining.")

# 2. Separate features (X) and target variable (y)
target_col = 'ap_target'
X = df.drop(columns=[target_col])
y = df[target_col]

# [Optimization 2]: Apply log transformation to the target variable 
# (log1p handles 0 values safely)
y_log = np.log1p(y)

# 3. Split into train and test sets (maintain time order using shuffle=False)
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, shuffle=False)
# Keep the original y_test for final metric evaluation
_, _, _, y_test_original = train_test_split(X, y, test_size=0.2, shuffle=False)

# 4. [Optimization 1]: Use RobustScaler to handle extreme outliers in space weather data
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. [Optimization 3]: Tune SVR parameters
print("Training the optimized SVR model... (This may take several minutes for 40k+ samples, please wait)")
svm_model = SVR(kernel='rbf', C=10.0, gamma='scale', epsilon=0.1)
svm_model.fit(X_train_scaled, y_train_log)

# 6. Make predictions and revert the log transformation
y_pred_log = svm_model.predict(X_test_scaled)
# Convert the logarithmic predictions back to the original Ap index scale
y_pred = np.expm1(y_pred_log) 

# 7. Evaluate model performance
mse = mean_squared_error(y_test_original, y_pred)
mae = mean_absolute_error(y_test_original, y_pred)
r2 = r2_score(y_test_original, y_pred)

print("\n--- Optimized Model Evaluation Results ---")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

# --- Custom Accuracy Metrics ---

# Method 1: Tolerance-based Accuracy (Regression)
# Assuming a prediction is "correct" if it falls within ±5 of the actual Ap index
tolerance = 5.0  
accurate_predictions = np.abs(y_test_original - y_pred) <= tolerance
custom_accuracy = np.mean(accurate_predictions)
print(f"Custom Regression Accuracy (Tolerance ±{tolerance}): {custom_accuracy * 100:.2f}%")

# Method 2: Threshold-based Classification Accuracy
# Assuming Ap > 30 is considered a geomagnetic storm (1), otherwise normal (0)
threshold = 30.0  
y_test_binary = (y_test_original > threshold).astype(int)
y_pred_binary = (y_pred > threshold).astype(int)

storm_accuracy = accuracy_score(y_test_binary, y_pred_binary)
print(f"Geomagnetic Storm Classification Accuracy (Ap > {threshold}): {storm_accuracy * 100:.2f}%")

Loading data: ../data/time_binned_dataset.csv
After dropping missing values, the dataset has 43621 samples remaining.
Training the optimized SVR model... (This may take several minutes for 40k+ samples, please wait)

--- Optimized Model Evaluation Results ---
Mean Squared Error (MSE): 340.7480
Mean Absolute Error (MAE): 7.1599
R2 Score: -0.0676
Custom Regression Accuracy (Tolerance ±5.0): 67.82%
Geomagnetic Storm Classification Accuracy (Ap > 30.0): 94.18%
